# 🎙️ Audio Filtering Pipeline for Indic Speech

**Goal:** Automatically detect and remove low-quality audio samples from large Indic speech datasets (e.g., IndicVoices) before training speech models.

## What this pipeline does
```
HuggingFace Stream
      │
      ▼
  Preprocessing  ──── resample → 16 kHz, mono, peak-normalise
      │
      ▼
 Layer 1: Signal Metrics   ── SNR, RMS, Clipping, Silence, Duration
      │
      ▼
 Layer 2: Spectral Metrics  ── Flatness, ZCR, Centroid
      │
      ▼
 Layer 3: Neural Metrics    ── Silero-VAD (speech ratio), DNSMOS (MOS score)
      │
      ▼
  Hard Rules + Scoring  ──── weighted score → KEEP / DISCARD
      │
      ▼
   results.csv + plots
```

**Runtime environment:** Google Colab (T4 GPU available, but pipeline is CPU-only)


---
## Step 1 — Environment Setup
Clone the repo and install dependencies.

In [ ]:
import os

REPO_URL = "https://github.com/Kartikgc9/Sarvam-Project.git"
REPO_DIR = "/content/Sarvam-Project"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
# Install all dependencies
!pip install -q -r requirements.txt
print("✅ Dependencies installed")

In [ ]:
# Download DNSMOS ONNX model and cache Silero-VAD
!python setup_models.py

---
## Step 2 — Understanding the Metrics

Before running the pipeline, let's **visualise each metric** on synthetic audio to build intuition.

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

from src.metrics.signal_metrics import (
    compute_snr, compute_silence_ratio,
    compute_clipping_ratio, compute_rms_energy
)
from src.metrics.spectral_metrics import (
    compute_spectral_flatness, compute_zcr
)

SR = 16_000

# --- Synthetic audio samples ---
t = np.linspace(0, 2.0, SR * 2, endpoint=False)

clean_speech  = (0.4 * np.sin(2*np.pi*200*t) +         # fundamental
                 0.2 * np.sin(2*np.pi*400*t) +         # 1st harmonic
                 0.1 * np.sin(2*np.pi*800*t)).astype(np.float32)

noisy_speech  = (clean_speech +
                 0.25 * np.random.default_rng(0).standard_normal(SR*2)).astype(np.float32)

clipped       = np.clip(clean_speech * 3.0, -1.0, 1.0).astype(np.float32)

silence       = np.zeros(SR * 2, dtype=np.float32)

white_noise   = (0.3 * np.random.default_rng(1).standard_normal(SR*2)).astype(np.float32)

samples = {
    'Clean Speech'  : clean_speech,
    'Noisy Speech'  : noisy_speech,
    'Clipped Speech': clipped,
    'Silence'       : silence,
    'White Noise'   : white_noise,
}

print("Synthetic audio samples created.")

In [ ]:
# ── Visualise waveforms and compute metrics for each sample ─────────────────
fig, axes = plt.subplots(len(samples), 1, figsize=(14, 12), sharex=True)

time_axis = np.linspace(0, 2.0, SR * 2)

for ax, (name, audio) in zip(axes, samples.items()):
    snr      = compute_snr(audio, SR)
    silence  = compute_silence_ratio(audio, SR)
    clipping = compute_clipping_ratio(audio)
    flatness = compute_spectral_flatness(audio, SR)

    ax.plot(time_axis, audio, lw=0.6, color='steelblue')
    ax.set_ylabel(name, fontsize=9)
    ax.set_ylim(-1.2, 1.2)
    ax.set_title(
        f"{name}  │  SNR={snr:.1f}dB  "
        f"Silence={silence:.2f}  Clipping={clipping:.3f}  Flatness={flatness:.3f}",
        fontsize=9, loc='left'
    )

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Waveforms + Metrics for Synthetic Audio', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("\nKey insight: Clean speech has high SNR, low silence, low clipping, low flatness.")
print("White noise has HIGH flatness (spectrum is uniform = no speech structure).")

In [ ]:
# ── Metric summary table ─────────────────────────────────────────────────────
import pandas as pd
from omegaconf import OmegaConf
from src.metrics.signal_metrics import compute_all_signal_metrics
from src.metrics.spectral_metrics import compute_all_spectral_metrics

cfg = OmegaConf.load('config/config.yaml')
rows = []
for name, audio in samples.items():
    sig  = compute_all_signal_metrics(audio, SR, cfg)
    spec = compute_all_spectral_metrics(audio, SR)
    rows.append({'Sample': name, **sig, **spec})

df_demo = pd.DataFrame(rows).set_index('Sample')
df_demo = df_demo[['snr_db','rms_energy','silence_ratio','clipping_ratio',
                    'spectral_flatness','zcr']].round(4)
df_demo.style.background_gradient(cmap='RdYlGn', axis=0)

---
## Step 3 — VAD and DNSMOS Demo

These are the **neural quality metrics** — more powerful than raw signal metrics.

In [ ]:
from src.metrics.vad_metrics import load_vad_model, compute_vad_ratio

vad_model, vad_utils = load_vad_model('models/')

print("VAD Ratios (fraction of audio with actual speech):")
print("-" * 45)
for name, audio in samples.items():
    vad = compute_vad_ratio(audio, SR, cfg, vad_model, vad_utils)
    bar = '█' * int(vad * 20) + '░' * (20 - int(vad * 20))
    print(f"  {name:<20} {bar}  {vad:.3f}")

In [ ]:
from src.metrics.dnsmos import load_dnsmos_model, compute_dnsmos

dnsmos_session = load_dnsmos_model('models/sig_bak_ovr.onnx')

print("DNSMOS P.835 Scores (1=bad, 5=excellent):")
print("-" * 55)
print(f"  {'Sample':<22} {'SIG':>6} {'BAK':>6} {'OVR':>6}")
print("-" * 55)
for name, audio in samples.items():
    d = compute_dnsmos(audio, SR, dnsmos_session)
    print(f"  {name:<22} {d['dnsmos_sig']:>6.2f} {d['dnsmos_bak']:>6.2f} {d['dnsmos_ovr']:>6.2f}")

---
## Step 4 — Scoring Demo

Show how the **two-stage filtering** works: hard rules first, then weighted score.

In [ ]:
from src.scoring import make_decision

print("Scoring synthetic samples (threshold = 0.60):")
print("=" * 70)

for name, audio in samples.items():
    sig  = compute_all_signal_metrics(audio, SR, cfg)
    spec = compute_all_spectral_metrics(audio, SR)
    vad  = compute_vad_ratio(audio, SR, cfg, vad_model, vad_utils)
    dns  = compute_dnsmos(audio, SR, dnsmos_session)

    all_metrics = {**sig, **spec, 'vad_ratio': vad, **dns}

    # Need to add duration separately
    all_metrics['duration'] = len(audio) / SR

    result = make_decision(all_metrics, cfg)

    icon = '✅' if result['decision'] == 'KEEP' else '❌'
    print(f"  {icon} {name:<22}  score={result['quality_score']:.3f}  "
          f"→ {result['decision']}  [{result['reason']}]")

---
## Step 5 — Run Pipeline on IndicVoices

Now we run the **real pipeline** on a subset of the IndicVoices dataset from HuggingFace.

> **Note:** If IndicVoices requires HuggingFace authentication, run the cell below first.

In [ ]:
# Optional: log in to HuggingFace if the dataset requires authentication
# from huggingface_hub import login
# login()   # Will prompt for your HF token

In [ ]:
# Run on 50 samples per language (quick demo — takes ~3-5 mins)
# Change --max-samples to process more
!python main.py --max-samples 50

---
## Step 6 — Results and Visualizations

In [ ]:
import pandas as pd

df = pd.read_csv('outputs/results.csv')
print(f"Total samples processed: {len(df)}")
print(f"Kept:      {(df['decision']=='KEEP').sum()} ({(df['decision']=='KEEP').mean()*100:.1f}%)")
print(f"Discarded: {(df['decision']=='DISCARD').sum()}")
print()
df.head(15)

In [ ]:
# Generate all plots
!python main.py --visualize-only

In [ ]:
from IPython.display import Image, display
import glob

plot_files = sorted(glob.glob('outputs/plots/*.png'))
for f in plot_files:
    print(f"\n{'='*60}")
    print(f"  {f}")
    print('='*60)
    display(Image(f))

In [ ]:
import json
with open('outputs/plots/summary.json') as f:
    summary = json.load(f)

print(json.dumps(summary, indent=2))

---
## Step 7 — Per-Language Analysis

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv('outputs/results.csv')

# Average metrics per language for KEPT samples
kept_df = df[df['decision'] == 'KEEP']

lang_metrics = kept_df.groupby('language')[[
    'snr_db', 'vad_ratio', 'dnsmos_ovr', 'quality_score'
]].mean().round(3)

print("Average metrics for KEPT samples, per language:")
print(lang_metrics.to_string())

# Plot
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
cols = ['snr_db', 'vad_ratio', 'dnsmos_ovr', 'quality_score']
titles = ['Avg SNR (dB)', 'Avg VAD Ratio', 'Avg DNSMOS OVR', 'Avg Quality Score']

for ax, col, title in zip(axes, cols, titles):
    if col in lang_metrics.columns:
        lang_metrics[col].plot(kind='bar', ax=ax, color='#3498db', alpha=0.8)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xlabel('Language')
        ax.tick_params(axis='x', rotation=45)

plt.suptitle('Quality Metrics by Language (KEPT samples only)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 8 — Scalability Analysis

Demonstrating how the pipeline scales to millions of samples.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor

from src.metrics.signal_metrics import compute_all_signal_metrics
from src.metrics.spectral_metrics import compute_all_spectral_metrics
from omegaconf import OmegaConf

cfg = OmegaConf.load('config/config.yaml')
SR  = 16_000

def time_single_sample(duration_s=3.0):
    """Measure time to compute CPU metrics for one sample."""
    audio = (0.4 * np.sin(2*np.pi*200*np.linspace(0, duration_s, int(duration_s*SR)))).astype(np.float32)
    t0 = time.perf_counter()
    compute_all_signal_metrics(audio, SR, cfg)
    compute_all_spectral_metrics(audio, SR)
    return time.perf_counter() - t0

# Benchmark
n_runs = 20
times  = [time_single_sample() for _ in range(n_runs)]
avg_ms = np.mean(times) * 1000
std_ms = np.std(times) * 1000

print(f"CPU metrics per sample: {avg_ms:.1f} ± {std_ms:.1f} ms")

# Project throughput
samples_per_sec = 1000 / avg_ms
samples_per_hour = samples_per_sec * 3600

print(f"\nThroughput (single core):")
print(f"  {samples_per_sec:.1f} samples/sec")
print(f"  {samples_per_hour/1000:.0f}k samples/hour")

print(f"\nProjected throughput (4 workers):")
print(f"  {samples_per_sec*4:.1f} samples/sec")
print(f"  ~{samples_per_hour*4/1_000_000:.1f}M samples in 8 hours")
print(f"\n→ IndicVoices (1000 hrs at avg 5s/clip ≈ 720k samples)")
print(f"  Estimated full-run time: ~{720_000 / (samples_per_sec*4) / 3600:.1f} hours")

---
## Step 9 — Architecture Explanation (for Live Demo)

Key points to explain during the demo:

### Why these metrics?
| Metric | Maps to | Threshold |
|---|---|---|
| SNR | Noise level — noisy recordings break ASR | > 10 dB |
| VAD Ratio | Actual speech present, not just audio | > 0.3 |
| DNSMOS | Human-perceived quality (1-5 MOS scale) | > 2.0 |
| Silence Ratio | Recording content vs. dead air | < 0.8 |
| Clipping | Mic saturation / overdriven ADC | < 5% |
| Spectral Flatness | Distinguishes speech from broadband noise | context |

### Why these weights?
```python
score = 0.30 * SNR        # #1 enemy of ASR is noise
      + 0.25 * VAD        # no speech = useless sample
      + 0.20 * DNSMOS     # perceptual quality beats physical metrics
      + 0.15 * active     # penalise mostly-silent recordings
      + 0.10 * no_clip    # signal integrity
```

### Why streaming?
> IndicVoices has 7000+ hours of audio. Loading it fully = ~84 GB RAM. 
> Streaming processes one sample at a time — no upper memory limit.

### Why multiprocessing?
> Python's GIL blocks true parallel CPU execution in threads.
> ProcessPoolExecutor spawns separate OS processes — each gets a full CPU core.
> For 4 workers, throughput is ~3.5× faster than single-threaded.

In [ ]:
# ── Live walkthrough of one sample end-to-end ────────────────────────────────
import numpy as np
from omegaconf import OmegaConf
from src.preprocess import preprocess_audio
from src.metrics.signal_metrics import compute_all_signal_metrics
from src.metrics.spectral_metrics import compute_all_spectral_metrics
from src.metrics.vad_metrics import compute_vad_ratio
from src.metrics.dnsmos import compute_dnsmos
from src.scoring import make_decision

cfg = OmegaConf.load('config/config.yaml')
SR  = 16_000

# Simulate a realistic speech sample
rng   = np.random.default_rng(42)
t     = np.linspace(0, 4.0, SR*4, endpoint=False)
speech = (
    0.3*np.sin(2*np.pi*150*t) + 0.2*np.sin(2*np.pi*300*t) +
    0.1*np.sin(2*np.pi*600*t) + 0.05*rng.standard_normal(SR*4)
).astype(np.float32)

# Add silence padding (simulate real recordings)
speech[:SR//2]  = 0.0
speech[-SR//2:] = 0.0

sample = {
    'id': 'demo_hi_000001',
    'language': 'hi',
    'audio_array': speech,
    'sample_rate': SR,
    'transcript': 'नमस्ते',
    'speaker_id': 'spk_001',
}

print("=" * 60)
print("  End-to-end processing of one sample")
print("=" * 60)

# Step 1: Preprocess
sample = preprocess_audio(sample, target_sr=SR)
audio  = sample['audio_array']
print(f"\n[1] Preprocessing")
print(f"    Duration : {sample['duration']:.2f}s")
print(f"    Shape    : {audio.shape}")
print(f"    Peak     : {np.max(np.abs(audio)):.4f} (normalised)")

# Step 2: Signal metrics
sig = compute_all_signal_metrics(audio, SR, cfg)
print(f"\n[2] Signal Metrics")
for k, v in sig.items():
    print(f"    {k:<20} : {v:.4f}")

# Step 3: Spectral metrics
spec = compute_all_spectral_metrics(audio, SR)
print(f"\n[3] Spectral Metrics")
for k, v in spec.items():
    print(f"    {k:<20} : {v:.4f}")

# Step 4: VAD
vad = compute_vad_ratio(audio, SR, cfg, vad_model, vad_utils)
print(f"\n[4] VAD Ratio          : {vad:.4f}")

# Step 5: DNSMOS
dns = compute_dnsmos(audio, SR, dnsmos_session)
print(f"\n[5] DNSMOS Scores")
for k, v in dns.items():
    print(f"    {k:<20} : {v:.3f}  (1=bad, 5=excellent)")

# Step 6: Decision
all_metrics = {**sig, **spec, 'vad_ratio': vad, **dns, 'duration': sample['duration']}
decision    = make_decision(all_metrics, cfg)
print(f"\n[6] Final Decision")
print(f"    Quality Score  : {decision['quality_score']:.4f}")
print(f"    Decision       : {'✅ KEEP' if decision['decision']=='KEEP' else '❌ DISCARD'}")
print(f"    Reason         : {decision['reason']}")

---
## Summary

This pipeline demonstrates:

| Aspect | Approach |
|---|---|
| **Scalability** | HuggingFace streaming + ProcessPoolExecutor — works on any dataset size |
| **Memory safety** | Single-file processing, incremental CSV writes, checkpoint recovery |
| **Metric depth** | 3 layers: signal → spectral → neural (VAD + DNSMOS) |
| **Justification** | Each metric traces to a real failure mode in speech data |
| **Configurability** | All thresholds/weights in `config/config.yaml` — no code changes needed |
| **Reproducibility** | Deterministic pipeline, version-controlled, tested |

**Bonus features implemented:** Language ID via dataset metadata, DNSMOS perceptual scoring, visual analysis dashboard.
